# 📄 NCERT PDF Text Extraction & Cleaning Pipeline
 
**Dataset:** NCERT Class 9 Science – Chapter 7: Motion (`iesc107.pdf`)  
**Goal:** Extract clean, structured, NLP-ready text from a multi-column scanned PDF.

---

## 🗺️ Pipeline Overview

This notebook walks through a complete text-extraction and cleaning pipeline, applied to an NCERT PDF that has multi-column layouts, broken headings, figure captions, and noisy formatting artifacts.

The pipeline is divided into **7 steps**:

| Step | What it does | Output file |
|------|-------------|-------------|
| 1 | Column-wise PDF text extraction | `output.txt` |
| 2 | Fix broken headings | *(in-memory)* |
| 3 | Full text cleaning pipeline | `cleaned_text.txt` |
| 4 | Remove figure references & noise | `cleaned_text.txt` (updated) |
| 5 | Separate Activities & Think-and-Act sections | `activities.txt`, `thinkandact.txt`, `final.txt` |
| 6 | Extract Questions section | `Questions.txt`, `final_cleaned_text.txt` |
| 7 | Extract Examples + fix formulas + final cleanup | `Examples.txt`, `informative_text.txt` |

> **Note:** Run all cells **top to bottom** in order — each step depends on the output of the previous one.

---

## ⚙️ Setup

Install the required library before running anything else.  
`pdfplumber` is used throughout this notebook for PDF reading and page-level text extraction.

In [1]:
!pip install pdfplumber

---

## Step 1 — Column-wise PDF Text Extraction

### 🔍 Problem
NCERT PDFs use a **two-column layout**. When you extract text naively (with default settings), pdfplumber reads left-to-right *across* both columns on each line. This breaks sentence boundaries:

```
"uniform motion. rest or in A body is in" ← garbled output
```

### ✅ Solution
Each page is split into a **left half** and a **right half** using bounding boxes. Text is extracted from each column separately and then concatenated in the correct reading order.

### 📤 Output
Saves the extracted raw text to `output.txt`.

In [2]:
import pdfplumber

pdf_path = "iesc107_removed (1).pdf"

final_text = ""

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        width = page.width
        height = page.height

        # Define column regions
        left_bbox = (0, 0, width/2, height)
        right_bbox = (width/2, 0, width, height)

        # Crop columns
        left_column = page.crop(left_bbox)
        right_column = page.crop(right_bbox)

        # Extract text in correct reading order
        left_text = left_column.extract_text() or ""
        right_text = right_column.extract_text() or ""

        final_text += left_text + "\n"
        final_text += right_text + "\n"

# Save output
with open("output.txt", "w", encoding="utf-8") as f:
    f.write(final_text)

print(final_text)

7
C
hapter
In everyday life, we see some objects at rest
and others in motion. Birds fly, fish swim,
blood flows through veins and arteries, and
cars move. Atoms, molecules, planets, stars
and galaxies are all in motion. We often
perceive an object to be in motion when its
position changes with time. However, there
are situations where the motion is inferred
through indirect evidences. For example, we
infer the motion of air by observing the
movement of dust and the movement of leaves
and branches of trees. What causes the
phenomena of sunrise, sunset and changing
of seasons? Is it due to the motion of the
earth? If it is true, why don’t we directly
perceive the motion of the earth?
An object may appear to be moving for
one person and stationary for some other. For
the passengers in a moving bus, the roadside
trees appear to be moving backwards. A
person standing on the road–side perceives
the bus alongwith the passengers as moving.
However, a passenger inside the bus sees his
fellow p

---

## Step 2 — Fix Broken Headings

### 🔍 Problem
PDF extraction often splits section headings across multiple lines. For example:

```
7.1 UNIFORM
MOTION
```
…instead of `7.1 UNIFORM MOTION`. It also produces character-level breaks like `U - NIFORM`.

### ✅ Solution
The `fix_split_headings_final()` function:
- Detects lines that start with a section number pattern like `7.x` or `7.x.x`
- Merges the heading with the following line(s) until it reaches 8 words or a new section
- Uses regex to clean broken patterns like `U - NIFORM` → `UNIFORM`

### 📝 Note
This function is **defined here** but applied later in Step 3 as part of the full cleaning pipeline.

In [3]:
import re

def fix_split_headings_final(text):
    lines = text.split("\n")
    fixed_lines = []
    i = 0
    n = len(lines)

    while i < n:
        line = lines[i].strip()

        # -----------------------------
        # Detect heading start: 7.x or 7.x.x
        # -----------------------------
        if re.match(r'^\d+\.\d+(\.\d+)?', line):
            combined = line

            j = i + 1

            # Merge next 1–2 lines (headings are short)
            while j < n and len(combined.split()) < 8:
                next_line = lines[j].strip()

                # stop if next is new section or empty
                if re.match(r'^\d+\.\d+', next_line):
                    break
                if next_line == "":
                    break

                combined += " " + next_line
                j += 1

            # Clean broken patterns inside heading
            combined = re.sub(r'([A-Z])\s*[-–]\s*([A-Z])', r'\1\2', combined)
            combined = re.sub(r'\b([A-Z])\s+([A-Z]+)', r'\1\2', combined)

            fixed_lines.append(combined)
            i = j
            continue

        # -----------------------------
        # Fix standalone broken words
        # -----------------------------
        if i + 1 < n:
            next_line = lines[i + 1].strip()

            if (
                len(line) == 1 and line.isalpha()
                and next_line.isalpha()
            ):
                combined = line + next_line
                fixed_lines.append(combined)
                i += 2
                continue

        fixed_lines.append(line)
        i += 1

    return "\n".join(fixed_lines)

---

## Step 3 — Full Text Cleaning Pipeline

### 🔍 Problem
Even after column-wise extraction, the raw text has multiple types of noise:
- **Repeated characters:** `MMMOTION` instead of `MOTION`
- **Spaced-out characters:** `M O T I O N` instead of `MOTION`
- **Broken chapter headings:** `7` / `C` / `hapter` on separate lines
- **Misplaced words:** The word `MOTION` appears far away from its chapter header

### ✅ Solution — Four helper functions

| Function | What it fixes |
|---|---|
| `fix_repeated_chars()` | `MMMOTION` → `MOTION` using regex backreference |
| `fix_spaced_words()` | `M O T I O N` → `MOTION` by detecting spaced letter sequences |
| `fix_heading_lines()` | Reassembles broken `Chapter N` headers across 2–3 lines |
| `fix_chapter_motion_position()` | Moves the misplaced `MOTION` title to immediately follow the Chapter header |

These four functions are **defined** in the cell below. The next cell applies them in sequence.

In [4]:
import pdfplumber
import re

def fix_repeated_chars(text):
    return re.sub(r'(.)\1{2,}', r'\1', text)

def fix_spaced_words(text):
    return re.sub(r'\b(?:[A-Za-z]\s){2,}[A-Za-z]\b',
                  lambda m: m.group(0).replace(" ", ""),
                  text)

def fix_heading_lines(text):
    lines = text.split("\n")
    fixed_lines = []
    i = 0

    while i < len(lines):
        line = lines[i].strip()

        # Case 1: Chapter number + broken "Chapter"
        if (
            i + 2 < len(lines)
            and lines[i].strip().isdigit()
            and lines[i+1].strip().lower() in ["c", "ch"]
            and re.match(r"hapter", lines[i+2].strip(), re.IGNORECASE)
        ):
            chapter_num = lines[i].strip()
            fixed_lines.append(f"Chapter {chapter_num}")
            i += 3
            continue

        # Case 2: Broken ALL CAPS word like M + OTION
        if (
            i + 1 < len(lines)
            and len(lines[i].strip()) == 1
            and lines[i].strip().isalpha()
            and lines[i+1].strip().isalpha()
        ):
            combined = lines[i].strip() + lines[i+1].strip()
            fixed_lines.append(combined)
            i += 2
            continue

        # Default: keep line
        fixed_lines.append(line)
        i += 1

    return "\n".join(fixed_lines)

def extract_clean_text(pdf_path):
    all_text = ""

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text(x_tolerance=2, y_tolerance=2)

            if text:
                text = fix_repeated_chars(text)
                text = fix_spaced_words(text)
                text = fix_heading_lines(text) # Apply the heading fix
                text = fix_chapter_motion_position(text)
                all_text += text + "\n"

    return all_text

### `fix_chapter_motion_position()` — Repositioning the Chapter Title

This helper function handles a specific artifact from the PDF: the word `MOTION` (the chapter title) gets extracted far from the `Chapter 7` line due to how the PDF is laid out.

**What it does:**
1. Scans all lines and captures the `MOTION` line, removing it from its original position
2. Re-inserts it immediately after the line that matches `Chapter N`

This function is used inside `extract_clean_text()` defined above.

In [5]:
def fix_chapter_motion_position(text):
    lines = text.split("\n")

    cleaned_lines = []
    motion_line = None

    # Step 1: normalize MOTION (already repeated chars fixed earlier)
    for line in lines:
        stripped = line.strip()

        # detect MOTION (after repeated char fix it becomes "MOTION")
        if stripped.upper() == "MOTION":
            motion_line = "MOTION"
            continue

        # remove junk like MMMMM, etc. (already mostly handled)
        if re.match(r'^[A-Z]{2,}$', stripped) and stripped != "MOTION":
            continue

        cleaned_lines.append(line)

    # Step 2: insert MOTION after Chapter line
    final_lines = []
    inserted = False

    for line in cleaned_lines:
        final_lines.append(line)

        if not inserted and re.match(r'^Chapter\s*\d+', line, re.IGNORECASE):
            if motion_line:
                final_lines.append(motion_line)
                inserted = True

    return "\n".join(final_lines)

### 🚀 Apply the Cleaning Pipeline

Now all four functions are applied to `final_text` (from Step 1) in a logical order:

1. `fix_repeated_chars` — character-level noise first
2. `fix_spaced_words` — word-spacing issues
3. `fix_heading_lines` — structural heading repair
4. `fix_split_headings_final` — section number heading repair
5. `fix_chapter_motion_position` — title repositioning

The cleaned result is printed. (Saving to file is commented out — uncomment to save.)

In [6]:
# Apply cleaning on column-extracted text
text = final_text

text = fix_repeated_chars(text)
text = fix_spaced_words(text)
text = fix_heading_lines(text)
text = fix_split_headings_final(text)
text = fix_chapter_motion_position(text)

# Save cleaned output
#with open("cleaned_output.txt", "w", encoding="utf-8") as f:
    #f.write(text)

print(text)

Chapter 7
MOTION
In everyday life, we see some objects at rest
and others in motion. Birds fly, fish swim,
blood flows through veins and arteries, and
cars move. Atoms, molecules, planets, stars
and galaxies are all in motion. We often
perceive an object to be in motion when its
position changes with time. However, there
are situations where the motion is inferred
through indirect evidences. For example, we
infer the motion of air by observing the
movement of dust and the movement of leaves
and branches of trees. What causes the
phenomena of sunrise, sunset and changing
of seasons? Is it due to the motion of the
earth? If it is true, why don’t we directly
perceive the motion of the earth?
An object may appear to be moving for
one person and stationary for some other. For
the passengers in a moving bus, the roadside
trees appear to be moving backwards. A
person standing on the road–side perceives
the bus alongwith the passengers as moving.
However, a passenger inside the bus sees his
fe

---

## Step 4 — Remove Figure References & Non-Contextual Noise

### 🔍 Problem
The extracted text contains several categories of noise that are harmful for NLP tasks:

- **Bullet/symbol characters** like `●`, `▪`, `►` that don't carry meaning
- **Reprint lines** and **standalone year numbers** from page headers/footers (e.g. `Reprint 2024`, `2025-26`)
- **Figure caption lines** like `Fig. 7.1: Motion of a car` — entire lines to be dropped
- **Inline figure references** like `(Fig. 7.3)` embedded mid-sentence
- **Broken dangling phrases** like `shown in .` left behind after figure removal

### ✅ Solution — `preprocess_text()` function

The function handles all six categories in a deliberate order, processing symbol removal first, then line-level filtering, then inline cleanup, and finally whitespace normalisation.

### 📤 Output
Saves the cleaned text to `cleaned_text.txt`.

In [7]:
import re

def preprocess_text(text):

    # -----------------------------
    # 1. Remove bullets & symbols
    # -----------------------------
    text = re.sub(r'[•●▪■▪◦◆►]', '', text)
    text = re.sub(r'[_]', '', text)

    # -----------------------------
    # 2. Remove Reprint / year lines
    # -----------------------------
    text = re.sub(r'Reprint\s*\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b\d{4}-\d{2}\b', '', text)   # 2025-26
    text = re.sub(r'\b\d{4}\b', '', text)         # standalone year
    
    # -----------------------------
    # 3. Remove figure captions (FULL LINES)
    # -----------------------------
    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:
        stripped = line.strip()

        # Remove lines like:
        # Fig. 7.1: ...
        # Figure 7.2 ...
        if re.match(r'^(Fig\.?|Figure)\s*\d+(\.\d+)?', stripped, re.IGNORECASE):
            continue

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)
    
    # -----------------------------
    # 4. Remove inline figure refs
    # -----------------------------
    text = re.sub(r'\(?\bFig\.?\s*\d+(\.\d+)?[a-zA-Z]?\)?', '', text, flags=re.IGNORECASE)

    # -----------------------------
    # 5. Clean leftover broken phrases
    # -----------------------------
    text = re.sub(r'\b(in|shown in|given in)\s*\.', '', text)

    # -----------------------------
    # 6. Clean spacing (KEEP paragraphs)
    # -----------------------------
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    text = "\n".join(lines)

    return text

### 🚀 Run the Preprocessing

Applies `preprocess_text()` to the cleaned text from Step 3 and saves the result.

In [8]:
clean_text = preprocess_text(text)

# Save output
with open("cleaned_text.txt", "w", encoding="utf-8") as f:
    f.write(clean_text)

print(clean_text)

Chapter 7
MOTION
In everyday life, we see some objects at rest
and others in motion. Birds fly, fish swim,
blood flows through veins and arteries, and
cars move. Atoms, molecules, planets, stars
and galaxies are all in motion. We often
perceive an object to be in motion when its
position changes with time. However, there
are situations where the motion is inferred
through indirect evidences. For example, we
infer the motion of air by observing the
movement of dust and the movement of leaves
and branches of trees. What causes the
phenomena of sunrise, sunset and changing
of seasons? Is it due to the motion of the
earth? If it is true, why don’t we directly
perceive the motion of the earth?
An object may appear to be moving for
one person and stationary for some other. For
the passengers in a moving bus, the roadside
trees appear to be moving backwards. A
person standing on the road–side perceives
the bus alongwith the passengers as moving.
However, a passenger inside the bus sees his
fe

### 📊 Output Analysis

After running Steps 1–4:

| Check | Result |
|---|---|
| Figure captions removed | ✅ |
| Inline figure references removed | ✅ |
| Bullet/symbol characters removed | ✅ |
| Paragraph structure preserved | ✅ |
| Conceptual content intact | ✅ |

The text is now suitable for tokenization and retrieval model input.

---

## 📁 Data Structuring Strategy

From Step 5 onward, the approach shifts from *cleaning* to *separating*.

The NCERT chapter contains several types of content that are **not useful for a knowledge retrieval system** — Activities, Think and Act boxes, Questions, and Examples. Rather than discarding these sections, each type is **extracted into its own file** so they can be reused later (e.g. for building an evaluation dataset or a QA training set).

| Section type | Saved to |
|---|---|
| Activities | `activities.txt` |
| Think and Act | `thinkandact.txt` |
| Questions | `Questions.txt` |
| Examples | `Examples.txt` |
| What You Have Learnt | `what_you_have_learnt.txt` |
| Chapter End Exercises | `Chapter End Exercises.txt` |
| **Core informative text** | **`informative_text.txt`** |

---

## Step 5 — Separate Activities & Think-and-Act Sections

### 🔍 Problem
Activity blocks and Think-and-Act boxes are interspersed throughout the chapter. They don't contain factual knowledge — they are procedural or reflective prompts — and including them in the main text would add noise to retrieval results.

### ✅ Solution — `extract_sections()` function

The function iterates through lines and uses regex to detect the start of each block type:
- `Activity N.N` → collected into `activities`
- `Think and Act` → collected into `think_act`
- Everything else → kept in `main_text`

**Stop condition:** Each block ends when a new section header, a new Activity, or a blank boundary is encountered.

### 📤 Output
- `activities.txt` — all Activity blocks
- `thinkandact.txt` — all Think and Act blocks
- `final.txt` — main text with those sections removed

In [9]:
import re

def extract_sections(text):
    lines = text.split("\n")

    main_text = []
    activities = []
    think_act = []

    i = 0
    n = len(lines)

    while i < n:
        line = lines[i].strip()

        # -----------------------------
        # ACTIVITY BLOCK
        # -----------------------------
        if re.match(r'^Activity\s*\d+(\.\d+)?', line, re.IGNORECASE):
            temp = [line]
            i += 1

            while i < n:
                next_line = lines[i].strip()

                # STOP conditions (very important)
                if (
                    re.match(r'^(Activity|Think and Act|Questions|Example|\d+\.\d+)', next_line, re.IGNORECASE)
                ):
                    break

                temp.append(next_line)
                i += 1

            activities.append("\n".join(temp))
            continue

        # -----------------------------
        # THINK AND ACT BLOCK
        # -----------------------------
        if re.match(r'^Think and Act', line, re.IGNORECASE):
            temp = [line]
            i += 1

            while i < n:
                next_line = lines[i].strip()

                if re.match(r'^(Activity|Questions|Example|\d+\.\d+)', next_line, re.IGNORECASE):
                    break

                temp.append(next_line)
                i += 1

            think_act.append("\n".join(temp))
            continue

        # -----------------------------
        # NORMAL CONTENT
        # -----------------------------
        main_text.append(line)
        i += 1

    return "\n".join(main_text), activities, think_act

### 🚀 Run the Extraction

Loads `cleaned_text.txt`, runs `extract_sections()`, and saves each part to its own file.

In [19]:
# Step 1: Load cleaned text
with open("cleaned_text.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Step 2: Extract sections
main_text, activities, think_act = extract_sections(text)

# Step 3: Save activities
with open("activities.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(activities))

# Step 4: Save Think & Act
with open("thinkandact.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(think_act))

# Step 5: Save cleaned main text
with open("final.txt", "w", encoding="utf-8") as f:
    f.write(main_text)

print(main_text)

Chapter 7
MOTION
In everyday life, we see some objects at rest
and others in motion. Birds fly, fish swim,
blood flows through veins and arteries, and
cars move. Atoms, molecules, planets, stars
and galaxies are all in motion. We often
perceive an object to be in motion when its
position changes with time. However, there
are situations where the motion is inferred
through indirect evidences. For example, we
infer the motion of air by observing the
movement of dust and the movement of leaves
and branches of trees. What causes the
phenomena of sunrise, sunset and changing
of seasons? Is it due to the motion of the
earth? If it is true, why don’t we directly
perceive the motion of the earth?
An object may appear to be moving for
one person and stationary for some other. For
the passengers in a moving bus, the roadside
trees appear to be moving backwards. A
person standing on the road–side perceives
the bus alongwith the passengers as moving.
However, a passenger inside the bus sees his
fe

---

## Step 6 — Extract the Questions Section

### 🔍 Problem
End-of-section "Questions" blocks appear throughout the chapter. Like Activities, they don't contribute to a knowledge retrieval system — but they are valuable for a future QA evaluation dataset.

### ✅ Solution
The code uses a **state machine** approach with a flag `in_questions`:
- When a line equals `Questions` (or the broken variant `Q uestions`), the flag is turned on
- All subsequent lines go into `questions_text` until a new section header is detected
- The `end_pattern` regex is carefully crafted to match section headers like `7.1 Describing` but **not** measurement values like `0.1 m s-2` (it requires an uppercase letter after the number)

### 📤 Output
- `Questions.txt` — all extracted questions
- `final_cleaned_text.txt` — main text without questions

In [11]:
import re

with open('final.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

questions_text = []
main_text = []

in_questions = False

# Regex perfectly targets section headers (e.g., "7.1 Describing", "7.1.1 MOTION"), Examples, and Activities.
# It requires an uppercase letter after the numbers, which prevents "0.1 m s-2" from triggering it!
end_pattern = re.compile(r'^(\d+\.\d+(\.\d+)?\s+[A-Z]|Example\s+\d+\.\d+|Activity\s+\d+\.\d+)')

for line in lines:
    stripped_line = line.strip()
    
    # Check if we are starting a questions block
    if stripped_line == 'Questions' or stripped_line == 'Q uestions':
        in_questions = True
        questions_text.append(line)
        continue
        
    if in_questions:
        # Check if we hit a new section header
        if end_pattern.match(stripped_line):
            in_questions = False
            main_text.append(line)
        # Check if it's a stray page number (just digits)
        elif re.match(r'^\d+$', stripped_line):
            main_text.append(line) # Put page numbers back in the main text
        else:
            questions_text.append(line)
    else:
        main_text.append(line)

with open('Questions.txt', 'w', encoding='utf-8') as f:
    f.writelines(questions_text)

with open('final_cleaned_text.txt', 'w', encoding='utf-8') as f:
    f.writelines(main_text)

print(f"Extraction complete!")
print(f"Extracted {len(questions_text)} lines of Questions and saved them to Questions.txt")
print(f"Main text without questions saved to final_cleaned_text.txt")


Extraction complete!
Extracted 93 lines of Questions and saved them to Questions.txt
Main text without questions saved to final_cleaned_text.txt


---

## Step 7 — Extract Examples, Fix Formulas & Final Cleanup

This is the most involved step, split across three sub-tasks.

---

### Step 7a — Extract Example Blocks

### 🔍 Problem
Worked examples (e.g. `Example 7.1`, `Example 7.2`) contain step-by-step calculations. These are useful for future QA training but dilute the core conceptual text for retrieval.

### ✅ Solution
A `start_pattern` regex detects lines like `Example 7.1`. Once inside an example block, lines are collected into `examples_text` until an `end_pattern` detects the start of the next real section.

**Bonus:** Standalone page numbers (single digit lines like `81`, `82`) are stripped at this stage too.

### 📤 Output
- `Examples.txt` — all worked examples
- `informative_text.txt` — main text, examples removed

---

### Step 7b — Fix Mangled Mathematical Formulas

### 🔍 Problem
When page numbers were stripped (they were used as denominators in fractions), mathematical formulas got broken. For example:

```
s
v = (7.1)          →  should be: v = s / t (7.1)
t
```

### ✅ Solution — `fix_mangled_formulas()` function

Targeted regex patterns repair each broken equation individually:
- Equations 7.1 through 7.8 are repaired with specific patterns
- Subscripts `t1`, `t2`, `s1`, `s2` pushed to wrong lines are pulled back
- Sideways y-axis text (`yticoleV`) leaked from graph images is removed
- Stray figure caption fragments are cleaned up

In [13]:
import re

with open('final_cleaned_text.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

examples_text = []
main_text = []
in_example = False

# Matches starts of examples like "Example 7.1"
start_pattern = re.compile(r'^(Example\s*\d+\.\d+|\d+\.\d+\s+Example)')

# Matches new sections (e.g., "7.1 Describing", "7.1.1 MOTION")
end_pattern = re.compile(r'^(\d+\.\d+(\.\d+)?\s+[A-Z]|Activity\s+\d+\.\d+)')

# 1. Separate Examples and strip stray page numbers
for line in lines:
    stripped_line = line.strip()
    
    # Ignore standalone numbers (removes page numbers like 1, 15, 81, 82)
    if re.match(r'^\d+$', stripped_line):
        continue
    
    if start_pattern.match(stripped_line):
        in_example = True
        examples_text.append(line)
        continue
        
    if in_example:
        if end_pattern.match(stripped_line):
            in_example = False
            main_text.append(line)
        else:
            examples_text.append(line)
    else:
        main_text.append(line)

with open('Examples.txt', 'w', encoding='utf-8') as f:
    f.writelines(examples_text)

# 2. Fix broken mathematical formulas and figure caption leaks
main_text_str = "".join(main_text)

def fix_mangled_formulas(text):
    # Fix 'average speed'
    text = re.sub(
        r'Total distance travelled\n\s*average speed =\n\s*Total time taken',
        r'average speed = Total distance travelled / Total time taken',
        text
    )
    # Fix 'average velocity' (the '2' denominator was cleared by the number filter)
    text = re.sub(
        r'initialvelocity\+finalvelocity\n\s*average velocity =',
        r'average velocity = (initial velocity + final velocity) / 2',
        text
    )
    # Fix Eq 7.1
    text = re.sub(r's\n\s*v = \(7\.1\)\n\s*t', r'v = s / t (7.1)', text)
    # Fix Eq 7.2
    text = re.sub(r'u\+v\n\s*v\n\s*Mathematically, = \(7\.2\)\n\s*av 2', r'Mathematically, v_av = (u + v) / 2 (7.2)', text)
    # Fix Eq 7.3
    text = re.sub(r'v[–\-]u\n\s*a =\n\s*\(7\.3\)\n\s*t', r'a = (v - u) / t (7.3)', text)
    # Fix Eq 7.4
    text = re.sub(r's [–\-]s\n\s*v 2 1\n\s*= \(7\.4\)\n\s*t [–\-]t\n\s*2 1', r'v = (s2 - s1) / (t2 - t1) (7.4)', text)
    # Fix Eq 7.8
    text = re.sub(r'2[π\w]r\n\s*v =\n\s*t\n\s*\(7\.8\)', r'v = 2πr / t (7.8)', text)
    
    # Fix linear equations
    text = re.sub(r's = ut \+ [½1/2]+\s*at2\s*\(7\.6\)', r's = ut + 1/2 at^2 (7.6)', text)
    text = re.sub(r'2\s*as = v2 [–\-] u2\s*\(7\.7\)', r'2as = v^2 - u^2 (7.7)', text)
    
    # --- CUSTOM GLITCH FIXES ---
    
    # Fix Area Formula (the '1' and '2' denominators were cleared by the number filter)
    text = re.sub(
        r's = area ABCDE\n\s*= area of the rectangle ABCD \+ area of\n\s*the triangle ADE\n\s*= AB [x×] BC \+ \(AD [x×] DE\)',
        r's = area ABCDE = area of the rectangle ABCD + area of the triangle ADE = AB × BC + 1/2 (AD × DE)',
        text
    )
    
    # Remove sideways y-axis text from graph
    text = re.sub(r'\)1[–\-]h\n\s*mk\(\n\s*yticoleV\n\s*', r'', text)
    
    # Fix subscripts t1, t2 that got pushed to the next line
    text = re.sub(r't and t([^\n]*?)\n\s*1\s*2', r't1 and t2\1', text)
    text = re.sub(r'\(t [–\-] t \)([^\n]*?)\n\s*2\s*1', r'(t2 - t1)\1', text)
    
    # Fix subscripts s1, s2 that got pushed to the next line
    text = re.sub(r'\(s [–\-] s \)([^\n]*?)\n\s*2\s*1', r'(s2 - s1)\1', text)
    
    # Fix acceleration formula and the floating 'v' variable
    text = re.sub(
        r'change in velocity\n\s*acceleration =\n\s*timetaken',
        r'acceleration = change in velocity / time taken',
        text
    )
    text = re.sub(
        r'If the velocity of an object changes from\n\s*v\n\s*an initial value u to the final value in time',
        r'If the velocity of an object changes from an initial value u to the final value v in time',
        text
    )

    # Strip stray figure captions
    text = re.sub(r'with uniform speed\n\s*We know that when an object travels equal', r'We know that when an object travels equal', text)
    text = re.sub(r'non-uniform speed\n\s*The distance-time graph for the motion', r'The distance-time graph for the motion', text)
    text = re.sub(r'of a car\n\s*is represented along the y-axis', r'is represented along the y-axis', text)

    return text

fixed_main_text = fix_mangled_formulas(main_text_str)

# Save to informative_text.txt
with open('informative_text.txt', 'w', encoding='utf-8') as f:
    f.write(fixed_main_text)

print(f"Extraction complete!")
print(f"Extracted {len(examples_text)} lines of Examples and saved them to Examples.txt")
print(f"Main text with PERFECTLY formatted formulas saved to informative_text.txt")


Extraction complete!
Extracted 176 lines of Examples and saved them to Examples.txt
Main text with PERFECTLY formatted formulas saved to informative_text.txt


---

### Step 7c — Final Cleanup & Recover Misplaced Paragraphs

### 🔍 Problem
After all the extraction steps, a few artifacts remain:

- Stray figure caption fragments that slipped through earlier filters
- A paragraph starting with `"If you carefully note, on being released..."` that got incorrectly placed inside `activities.txt` — it belongs in the main text
- A corrupted `v_av` subscript reference

### ✅ Solution — `final_text_cleanup()` function

- Removes each stray fragment using targeted `re.sub` or `.replace` calls
- Scans `activities.txt` for the misplaced paragraph using a regex with `re.DOTALL`
- Removes it from `activities.txt` and appends it to `informative_text.txt`

### 📤 Output
Updates `informative_text.txt` in place with all remaining issues resolved.

In [14]:
import re
import os

def final_text_cleanup():
    # 1. Read the informative_text.txt
    with open('informative_text.txt', 'r', encoding='utf-8') as f:
        info_text = f.read()

    # --- REMOVE STRAY FIGURE CAPTIONS & ARTIFACTS ---
    
    # Remove stray line: "n object on a straight line path"
    info_text = info_text.replace("n object on a straight line path\n", "")

    # Remove figure 7.2 caption fragment
    info_text = re.sub(
        r'\(a\) \(b\)\nLook at the situations\s*If\nthe bowling speed is 143 km h[–\-]1 in \(a\)\nwhat does it mean\? What do you understand\nfrom the signboard in \(b\)\?\n?', 
        '', 
        info_text
    )

    # Fix v_av subscript (where v is the average velocity -> where v_av)
    info_text = re.sub(
        r'where v is the average velocity, u is the initial\n\s*av',
        r'where v_av is the average velocity, u is the initial',
        info_text
    )

    # Remove stray caption: "uniform accelerations."
    info_text = info_text.replace("uniform accelerations.\n", "")

    # Remove Figure 7.7 caption fragment
    info_text = re.sub(
        r'uniformly accelerated motion\.\n\s*represents the motion of an object whose\n\s*velocity is decreasing with time while\n\s*representing the non-uniform variation of\n\s*velocity of the object with time\. Try to interpret\n\s*these graphs\.\n?',
        '',
        info_text
    )

    # Remove Figure 7.4/7.3 description fragment
    info_text = re.sub(
        r'The distance-time graph for the motion\n\s*of the car is\s*Note that the\n\s*shape of this graph is different from the earlier\n\s*distance-time graph\s*for uniform\n\s*motion\. The nature of this graph shows non-\n\s*linear variation of the distance travelled by\n\s*the car with time\. Thus, the graph shown in\n\s*speed\.\n?',
        '',
        info_text
    )

    # --- RECOVER MISSING PARAGRAPH FROM ACTIVITIES.TXT ---
    try:
        if os.path.exists('activities.txt'):
            with open('activities.txt', 'r', encoding='utf-8') as f:
                act_text = f.read()

            # Find the paragraph starting with "If you carefully note..."
            tangential_regex = re.compile(r'(If you carefully note, on being released.*?constant speed and so on\.)', re.DOTALL)
            match = tangential_regex.search(act_text)

            if match:
                tangential_content = match.group(1)
                
                # Remove it from activities.txt
                act_text = act_text.replace(tangential_content, '')
                
                # Clean up mangled 'What you have learnt' text at the end of activities.txt
                act_text = re.sub(r'e of position; it can be described in terms.*?at2\n\s*2\n?', '', act_text, flags=re.DOTALL)
                
                with open('activities.txt', 'w', encoding='utf-8') as f:
                    f.write(act_text.strip() + "\n")
                    
                # Append the tangential content to the correct place in informative_text.txt
                info_text += "\n" + tangential_content + "\n"
                print("Missing paragraph recovered from activities.txt and appended successfully!")
            else:
                print("Paragraph already recovered.")
    except Exception as e:
        print(f"Activities extraction issue: {e}")

    # Save the cleaned text back
    with open('informative_text.txt', 'w', encoding='utf-8') as f:
        f.write(info_text.strip() + "\n")

    print("Final text preprocessing complete!")

# Execute the cleanup
final_text_cleanup()


Missing paragraph recovered from activities.txt and appended successfully!
Final text preprocessing complete!


---

### Step 7d — Normalise Scientific Unit Notation

### 🔍 Problem
PDF extraction produces units like `m s-1` and `m s-2` using a Unicode en-dash (`–`) or a hyphen, which NLP tokenizers may misinterpret. For example, `m s-1` could be tokenized as `m`, `s`, `-`, `1` — four separate tokens — rather than as a single velocity unit.

### ✅ Solution — `format_superscript_units_for_nlp()`

Replaces patterns like `m s-1` and `km h-2` with standard ASCII caret notation:
- `m s-1` → `m s^-1`
- `km h-2` → `km h^-2`

This ensures tokenizers treat these as atomic units.

### 📤 Output
Updates `informative_text.txt` in place.

In [15]:
import re

def format_superscript_units_for_nlp():
    with open('informative_text.txt', 'r', encoding='utf-8') as f:
        text = f.read()

    # Replaces 'm s-1' with 'm s^-1' (Standard ASCII math notation)
    text = re.sub(r'\b(m|cm|km)\s+(s|h)[–\-]1\b', r'\1 \2^-1', text)
    
    # Replaces 'm s-2' with 'm s^-2'
    text = re.sub(r'\b(m|cm|km)\s+(s|h)[–\-]2\b', r'\1 \2^-2', text)
    
    with open('informative_text.txt', 'w', encoding='utf-8') as f:
        f.write(text)
        
    print("Units formatted safely for NLP tokenization!")

format_superscript_units_for_nlp()


Units formatted safely for NLP tokenization!


---

### Step 7e — Remove "What You Have Learnt" from Activities File

### 🔍 Problem
The section "What You Have Learnt" from the end of the chapter got mixed into `activities.txt` during the earlier extraction. It appears as a broken 3-line pattern:

```
What
you have
learnt
```

### ✅ Solution
The code reads `activities.txt` line by line and stops writing to the output as soon as it detects this exact 3-line pattern. Everything before it is saved to `activities_cleaned.txt`.

### 📤 Output
- `activities_cleaned.txt` — Activity blocks only, with "What You Have Learnt" removed

In [22]:
output_lines = []
buffer = []

with open("activities.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

i = 0
while i < len(lines):
    # Check for the 3-line pattern
    if (
        i + 2 < len(lines)
        and lines[i].strip() == "What"
        and lines[i + 1].strip() == "you have"
        and lines[i + 2].strip() == "learnt"
    ):
        break  # Stop processing from here onward

    output_lines.append(lines[i])
    i += 1

# Write back (or to a new file)
with open("activities_cleaned.txt", "w", encoding="utf-8") as f:
    f.writelines(output_lines)

---

## Step 8 — Extract "What You Have Learnt" & Chapter End Exercises

### 🔍 Problem
The final two sections of the chapter — **"What You Have Learnt"** (a summary) and **"Exercises"** (chapter-end questions) — need to be saved as separate files. These are useful as:
- Summary text for QA context
- An evaluation dataset of chapter-end questions

### ✅ Solution
The code re-opens the **original PDF** (`iesc107.pdf` — the full version without removed pages) and searches for:
- `"motion is a change of position"` as the anchor line inside the "What You Have Learnt" section (3 lines before it is the heading)
- `"exercises"` as the standalone line marking the start of the Exercises section

Once both anchors are found, the text is sliced at those positions and cleaned (SCIENCE headers and page numbers stripped).

### 📤 Output
- `what_you_have_learnt.txt` — chapter summary bullet points
- `Chapter End Exercises.txt` — all chapter-end exercise questions

In [17]:
import pdfplumber

pdf_path = "iesc107.pdf"

full_text = ""
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        extracted = page.extract_text()
        if extracted:
            full_text += extracted + "\n"

lines = full_text.splitlines()
start_idx = -1
exercises_idx = -1

for i, line in enumerate(lines):
    # We search line by line using simple python strings. No regex, no escape character bugs.
    if "motion is a change of position" in line.lower() and start_idx == -1:
        # The section heading 'What you have learnt' is usually 3 lines before this bullet point.
        start_idx = max(0, i - 3)
    if "exercises" in line.lower() and line.strip().lower() == "exercises" and start_idx != -1:
        exercises_idx = i
        break

if start_idx != -1:
    if exercises_idx != -1:
        learnt_lines = lines[start_idx:exercises_idx]
        exercises_lines = lines[exercises_idx:]
    else:
        learnt_lines = lines[start_idx:]
        exercises_lines = []

    # clean up the text (remove SCIENCE and page numbers)
    cleaned_learnt = [l for l in learnt_lines if "SCIENCE" not in l and "Reprint" not in l and not l.strip().isdigit()]
    cleaned_exercises = [l for l in exercises_lines if "SCIENCE" not in l and "Reprint" not in l and not l.strip().isdigit()]
    
    with open('what_you_have_learnt.txt', 'w', encoding='utf-8') as f:
        f.write("\n".join(cleaned_learnt).strip())
    print("Successfully extracted 'What you have learnt' section to what_you_have_learnt.txt!")
    
    if cleaned_exercises:
        with open('Chapter End Exercises.txt', 'w', encoding='utf-8') as f:
            f.write("\n".join(cleaned_exercises).strip())
        print("Successfully extracted 'Exercises' section to Chapter End Exercises.txt!")
else:
    print("Could not find the section.")


Successfully extracted 'What you have learnt' section to what_you_have_learnt.txt!
Successfully extracted 'Exercises' section to Chapter End Exercises.txt!


---

## ✅ Final Summary

### Files Produced

| File | Contents |
|---|---|
| `output.txt` | Raw column-extracted text (Step 1) |
| `cleaned_text.txt` | Fully cleaned text — noise removed (Steps 2–4) |
| `final.txt` | Cleaned text minus Activities & Think-and-Act (Step 5) |
| `final_cleaned_text.txt` | Above minus Questions section (Step 6) |
| `informative_text.txt` | **Core knowledge text** — final cleaned output (Step 7) |
| `activities.txt` | Extracted Activity blocks |
| `activities_cleaned.txt` | Activities without "What You Have Learnt" |
| `thinkandact.txt` | Extracted Think and Act blocks |
| `Questions.txt` | Extracted in-chapter questions |
| `Examples.txt` | Extracted worked examples |
| `what_you_have_learnt.txt` | Chapter summary section |
| `Chapter End Exercises.txt` | Chapter-end exercises |

### What Was Preserved
- Full conceptual/factual content of Chapter 7
- Mathematical formulas (repaired and normalised)
- Section structure and headings

### What Was Separated (not deleted)
- Activities, Think-and-Act, Questions, Examples, Summary, Exercises

---
*Pipeline complete. `informative_text.txt` is the clean, NLP-ready output.*